<div align="center">
  <img src="https://raw.githubusercontent.com/NaumanHSA/neurosurfer/main/docs/assets/banner/neurosurfer-banner-light.png" alt="Neurosurfer" width="45%"/>
</div>

<br/>

# 05 — Capstone: The Insight Engine

This is the **capstone**. Everything from the previous four notebooks comes together into one
graph that solves a real problem end-to-end: **point it at a sales database → get back a
professional Markdown insight report**, complete with charts.

The Insight Engine is a single [Graph](03_graph_agents.ipynb) that:

1. **Profiles** a synthetic sales dataset (a `function` node).
2. Fans out — *in parallel* — into three independent analyses:
   - **deterministic stats** with pandas (`function` node),
   - **charts** rendered to PNG (`function` node),
   - a **SQL analyst** that answers ad-hoc questions over a [SQLite MCP server](04_mcp_servers.ipynb) (`react` node).
3. **Reads the rendered dashboard with vision** — an LLM that actually *looks at the chart* (`react` node).
4. **Synthesises** an executive summary from all three streams (`react` node).
5. **Writes** `REPORT.md`, stitching the prose, the SQL findings, the metrics, and the charts together (`function` node).

Every concept in the series shows up here: [providers](01_providers_and_agents.ipynb),
[custom tools](02_custom_tools.ipynb), [graphs + parallelism](03_graph_agents.ipynb),
[MCP](04_mcp_servers.ipynb), and the newest piece — **vision**.

**What you'll learn:**

1. How to compose `function` and `react` nodes into a non-trivial DAG with a **parallel layer**.
2. How to give a `react` node **live MCP tools** by driving `GraphExecutor` directly.
3. How to make a node **see** — feed a chart image to a vision-capable model.
4. The five real-world **integration gotchas** that this code quietly solves (the actual teaching value — see the [Summary](#Summary)).

> The boilerplate lives in small supporting modules beside this notebook —
> [`database.py`](capstone/database.py) (the dataset), [`insight_nodes.py`](capstone/insight_nodes.py)
> (the custom tools), and [`insight_mcp_server.py`](capstone/insight_mcp_server.py) (the SQL server) —
> so the notebook can stay focused on wiring them into one graph.

---

## 1. Setup

This capstone needs a few optional extras — MCP and the plotting/vision stack:

```bash
pip install "neurosurfer[mcp]" pandas numpy matplotlib pillow
```

It also needs a **vision-capable** local model. The validated choice is **`qwen/qwen3.5-9b`**
in LM Studio — fast (~45s for the whole graph), clean output, no stalls. Load it and turn the
local server **on** (port 1234) before running.

> `google/gemma-4-12b-qat` also reads charts correctly, but it is much slower (300s+) and tends
> to *stall the stream* on a base node that emits reasoning-only content — so prefer Qwen here.

As in the earlier notebooks: point Python at the repo root (and at the `capstone/` folder so we
can import its reference modules), then define a notebook-friendly `IOHandler`.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, os, json, time, asyncio, sqlite3
from pathlib import Path

# Point Python at the repo root AND the capstone folder (so we can import the
# reference modules — insight_nodes.py, etc. — that ship beside this notebook).
NB_DIR    = Path(os.getcwd())            # tutorials/
REPO_ROOT = NB_DIR.parent
CAP       = NB_DIR / "capstone"
for p in (REPO_ROOT, CAP):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import neurosurfer
print(f"neurosurfer {neurosurfer.__version__}")

# Confirm the optional stack is present.
for mod in ("mcp", "pandas", "numpy", "matplotlib", "PIL"):
    try:
        __import__(mod)
        print(f"  {mod:11} OK")
    except ImportError:
        print(f"  {mod:11} MISSING — run: pip install 'neurosurfer[mcp]' pandas numpy matplotlib pillow")

# Where the dataset and generated artifacts go (both git-ignored under capstone/).
ART = CAP / "artifacts"; ART.mkdir(exist_ok=True)
DB  = CAP / "insight.db"

In [ ]:
# ── LM Studio connection (vision model) ───────────────────────────────────────
LM_STUDIO_URL   = "http://localhost:1234/v1"
LM_STUDIO_MODEL = "google/gemma-4-12b-qat"      # vision-capable; the validated choice
CONTEXT_WINDOW  = 48_000

from neurosurfer.llm.providers.openai import OpenAICompatProvider

provider = OpenAICompatProvider(
    model          = LM_STUDIO_MODEL,
    base_url       = LM_STUDIO_URL,
    api_key        = "lm-studio",
    context_window = CONTEXT_WINDOW,
)

# A local OpenAI-compatible model isn't auto-detected as vision-capable (we can't
# tell from the name), so we force the flag on. Make sure the loaded model really
# does accept images, or the vision node will fail.
provider.capabilities.supports_vision = True
print(f"Provider: {provider.model}  (supports_vision={provider.capabilities.supports_vision})")

In [ ]:
# ── matplotlib warm-up (GOTCHA #3) ────────────────────────────────────────────
# The charts node runs in a worker thread (it's in the parallel layer). The first
# matplotlib touch from a non-main thread can trip an IPython backend hook
# ("partially initialized module 'IPython'"). Warming it up once on the MAIN thread
# here avoids that. In a live Jupyter kernel IPython is already loaded so this is
# usually a no-op — but it's cheap insurance.
import matplotlib
matplotlib.use("Agg")
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg
FigureCanvasAgg(Figure())
print("matplotlib warmed up on the main thread.")

---

## 2. The pipeline at a glance

Seven nodes, wired as a DAG. The middle layer runs **in parallel**:

```
inputs {user_intent, db_path, artifacts_dir}
            │
            ▼
        loader            function · profile the dataset
            │
   ┌────────┼─────────────┐            ← PARALLEL layer (parallelism=3)
   ▼        ▼             ▼
 stats    charts      db_analyst       stats/charts = function (custom tools)
 (func)   (func)      (react: MCP)     db_analyst   = react + SQLite MCP tools
   │        │             │
   │        ▼             │
   │      vision          │            vision  = react + read_file (VISION on dashboard.png)
   │     (react)          │
   └────────┼─────────────┘
            ▼
        synthesis         react · executive summary (tools=[], plain text)
            │
            ▼
        report            function · assemble REPORT.md
```

| Node | Kind | What it does | From notebook |
|---|---|---|---|
| `loader` | `function` | Profile shape / range / totals | [02](02_custom_tools.ipynb) |
| `stats` | `function` | Deterministic pandas breakdowns | [02](02_custom_tools.ipynb) |
| `charts` | `function` | Render PNG charts + a combined dashboard | [02](02_custom_tools.ipynb) |
| `db_analyst` | `react` | Ad-hoc SQL over a SQLite **MCP** server | [04](04_mcp_servers.ipynb) |
| `vision` | `react` | **Looks at** the dashboard and describes it | this notebook |
| `synthesis` | `react` | Executive summary from all three streams | [01](01_providers_and_agents.ipynb) |
| `report` | `function` | Stitch everything into `REPORT.md` | [02](02_custom_tools.ipynb) |

The **parallelism** and DAG wiring are exactly what notebook [03](03_graph_agents.ipynb) taught —
we just point it at a bigger problem.

---

## 3. A dataset worth analysing

We generate a synthetic **sales** dataset into SQLite — ~2,000 orders across 18 months, five
product categories, four regions, three channels. It is seeded (`rng = default_rng(42)`) so the
report is reproducible, and it has two deliberate signals for the engine to discover:

- a gentle **upward revenue trend** plus a **Q4 seasonal bump**, and
- an **injected anomaly** — a viral Electronics/Phone spike in *South* during *2025-11* — that
  a good analysis should flag as worth investigating.

It is generated by `build_dataset`, which we import from [`database.py`](capstone/database.py).
This is the only "fake" part of the capstone; everything downstream treats it as a real database.

In [ ]:
from tutorials.capstone.database import build_dataset

build_dataset(DB)

---

## 4. Custom tools as `function` nodes

Three of our nodes are plain Python — no LLM. In a graph these are `function` nodes: each one
is a callable referenced by an `import_string` like `"insight_nodes:compute_stats"`, and the
engine invokes it as `fn(**{**graph_inputs, **dep_outputs})`. So every function takes `**kwargs`
and pulls out just the keys it needs; a dependency's output arrives under **its node id** (e.g.
the `synthesis` node's text shows up as the kwarg `synthesis=...`).

They live in [`capstone/insight_nodes.py`](capstone/insight_nodes.py) — we import them rather
than re-type 100 lines. The four callables:

| Callable | Node | Returns |
|---|---|---|
| `profile_dataset(db_path)` | `loader` | shape, date range, totals, head |
| `compute_stats(db_path)` | `stats` | revenue by region/category, monthly trend, MoM %, top products |
| `make_charts(db_path, artifacts_dir)` | `charts` | 3 PNGs **+ one combined `dashboard.png`** |
| `write_report(artifacts_dir, **deps)` | `report` | assembles `REPORT.md` |

Two real-world gotchas are baked into these functions:

- **GOTCHA #2 — don't render charts from a sandboxed `python_exec`.** The built-in code tool runs
  in a throwaway temp directory, so a file it writes never reaches your `cwd` and a later node
  can't read it. A `function` node runs **in-process**, writing straight to a known `artifacts/`
  path — which is why charting is a function node, not generated code.
- **GOTCHA #3 — thread-safe plotting.** `make_charts` uses the **object-oriented** matplotlib API
  (`Figure` + `FigureCanvasAgg`, no global `pyplot`) so it is safe to run in the parallel layer
  alongside the other nodes (paired with the main-thread warm-up from §1).

`make_charts` also renders **one combined `dashboard.png`** — that single image is all the vision
node reads (GOTCHA #5), keeping the multimodal context small and fast on a local VL model.

In [ ]:
from tutorials.capstone.insight_nodes import profile_dataset, compute_stats, make_charts, write_report

# These are just functions — call one directly to see what the loader node will emit.
print(profile_dataset(str(DB)))

---

## 5. The SQL analyst over MCP

The `db_analyst` node is a `react` agent whose tools are served by a tiny **read-only SQLite MCP
server** — [`capstone/insight_mcp_server.py`](capstone/insight_mcp_server.py). It exposes three
tools, exactly as you saw in notebook [04](04_mcp_servers.ipynb):

```python
@server.tool(annotations={"readOnlyHint": True})
def list_tables() -> str: ...           # what tables exist

@server.tool(annotations={"readOnlyHint": True})
def describe_table(table: str) -> str: ...   # column names + types

@server.tool(annotations={"readOnlyHint": True})
def run_sql(query: str) -> str: ...      # run a SELECT, return up to 50 rows
```

The agent runs a few `SELECT`s (totals by region, by category, the monthly trend) and writes its
findings as text. `run_sql` rejects anything that isn't a `SELECT`, so the database is safe.

### Connecting to MCP from a graph

An MCP `ClientSession` is bound to the event loop that opened it. The synchronous `GraphExecutor`
runs every `react` node in a new event loop (`asyncio.run` per node), so naively the session
would be called from a *different* loop than the one that created it — a bug that would cause it
to hang and time out.

`McpSession` handles this for you. It keeps the session alive on its own background loop and
`McpTool` auto-bridges every call back to that loop transparently. You never see it — you just
create a `McpSession`, call `.connect()`, hand its `.tools()` to a `ToolPool`, and everything
works across notebooks, scripts, and multi-threaded graph executors.


In [ ]:
from neurosurfer.config.mcp import McpServerConfig
from neurosurfer.mcp import McpSession


Connect to the SQLite MCP server (launched as a stdio subprocess with this same
Python interpreter, handed the path to our `insight.db`).


In [ ]:
session = McpSession([McpServerConfig(
    name="db",
    transport="stdio",
    command=sys.executable,
    args=[str(CAP / "insight_mcp_server.py"), str(DB)],
)])
for s in session.connect():
    print(f"server {s.name!r}: connected={s.connected}, tools={s.tools}")


---

## 6. Assemble the tool pool

One more wiring decision. Live MCP tools are deliberately **not** part of `workflow_node_tools()`,
so the high-level `WorkflowRunner` from notebook [03](03_graph_agents.ipynb) can't see them. To
give a `react` node live MCP tools we drive **`GraphExecutor` directly** and hand it an explicit
`ToolPool` + `ToolContext`.

The pool is the MCP tools (for `db_analyst`) plus the built-in **`read_file`** (the vision node
calls `read_file` on the dashboard path — that's how the image enters the conversation). Each
node only ever sees the subset named in its `tools=[...]` list.


In [ ]:
from neurosurfer.tools import default_pool, ToolPool, TerminalIOHandler
from neurosurfer.tools.base import ToolContext

# Pick the read-only `read_file` built-in and combine it with the MCP server's tools
# into one pool the tools can be called from.
builtins = default_pool().select(["read_file"]).all()
pool = ToolPool([*session.tools(), *builtins])
print("pool:", pool.names())

# ToolContext bundles everything a tool needs beyond its own args — here the working
# directory and an IOHandler.
#
# The IOHandler is how a tool talks to the human: `notify(...)` prints progress, and the
# approval methods (shell / write / plan gates) ask for a yes/no. TerminalIOHandler
# answers those on stdin/stdout — it prints notifications and, on a gated action, blocks
# with an input() prompt (works in a terminal and in a notebook alike).
#
# Note: we're calling tools *directly* through this raw ctx, not via an agent's
# permission loop, and both tools here are read-only (read_file + MCP reads) — so no gate
# actually fires. The handler is used only for notifications. If you'd rather the notebook
# never pause for input (e.g. "Run all"), swap in AutoApproveIOHandler.
ctx = ToolContext(cwd=CAP, io=TerminalIOHandler())


---

## 7. Build the graph

Now the DAG itself. Two more gotchas are encoded in the node goals and policies:

- **GOTCHA #4 — get clean text out of a `react` node.** A react node that ends by calling `finish`
  stashes its answer in `result.report`, but the graph reads `final_text` (the assistant *text*) —
  which would be empty. So the analysis nodes are told to **answer in plain text and not call
  `finish`**: the loop then ends naturally on a no-tool-call turn and `final_text` is populated.
  We also cap `max_new_tokens` via `NodePolicy` so a "thinking" local model doesn't burn minutes.
- **GOTCHA #5 — keep vision cheap.** The vision node reads **one** combined dashboard image (not
  three separate charts): far less multimodal context = faster and more reliable on small VL
  models. (Vision itself was switched on back in §1 via `supports_vision=True`.)

`NodePolicy` is per-node `temperature` / `max_new_tokens` / `timeout_s`. `depends_on` defines the
edges; `stats`, `charts`, and `db_analyst` all depend only on `loader`, so the engine runs them
concurrently.

In [ ]:
from neurosurfer.graph import Graph, GraphNode, GraphExecutor
from neurosurfer.graph.engine.schema import NodePolicy

def build_graph() -> Graph:
    return Graph(
        name="insight_engine",
        description="Profile a sales dataset, analyse it (stats + SQL + charts), read the charts with vision, and write a report.",
        nodes=[
            GraphNode(id="loader", kind="function", callable="tutorials.capstone.insight_nodes:profile_dataset",
                      description="Profile the dataset."),

            # ── parallel layer: all three depend only on loader ──────────────
            GraphNode(id="stats", kind="function", callable="tutorials.capstone.insight_nodes:compute_stats",
                      depends_on=["loader"], description="Deterministic pandas analytics."),
            GraphNode(id="charts", kind="function", callable="tutorials.capstone.insight_nodes:make_charts",
                      depends_on=["loader"], description="Render PNG charts."),
            GraphNode(id="db_analyst", kind="react", depends_on=["loader"],
                      tools=["list_tables", "describe_table", "run_sql"],
                      policy=NodePolicy(max_new_tokens=1200, temperature=0.2, timeout_s=180),
                      goal=("Use the SQL tools to answer the user's question with concrete numbers. "
                            "Run a few SELECT queries (totals by region, by category, and the monthly "
                            "trend). When you have the numbers, stop calling tools and write your findings "
                            "as your final reply — a short bulleted summary of the key figures.")),

            # ── vision reads the dashboard the charts node rendered ──────────
            GraphNode(id="vision", kind="react", depends_on=["charts"],
                      tools=["read_file"],
                      policy=NodePolicy(max_new_tokens=900, temperature=0.3, timeout_s=180),
                      goal=("The charts node printed a 'Dashboard image' path above. Call read_file ONCE on "
                            "that dashboard path to view it. Then stop calling tools and write your final "
                            "reply: describe the trend and any unusual spike you see across the three panels.")),

            # ── synthesis joins all three analysis streams (no tools) ────────
            GraphNode(id="synthesis", kind="react", tools=[],
                      depends_on=["stats", "db_analyst", "vision"],
                      policy=NodePolicy(max_new_tokens=1200, temperature=0.3, timeout_s=180),
                      goal=("Write a concise executive summary (<= 6 short paragraphs) of sales performance "
                            "from the stats, the SQL findings, and the chart observations above. Call out the "
                            "biggest drivers and any anomaly, and recommend where to focus next quarter. "
                            "Write the summary as your reply; do not call any tool.")),

            GraphNode(id="report", kind="function", callable="insight_nodes:write_report",
                      depends_on=["synthesis", "stats", "vision", "db_analyst", "charts"],
                      description="Assemble REPORT.md."),
        ],
        outputs=["report"],
    )

graph = build_graph()
print(f"graph {graph.name!r}: {len(graph.nodes)} nodes, outputs={graph.outputs}")

---

## 8. Run the Insight Engine

Hand the graph, the provider, the explicit `pool`, and the `ctx` to `GraphExecutor`, set
`parallelism=3`, and run. The inputs (`user_intent`, `db_path`, `artifacts_dir`) flow into every
`function` node as kwargs and seed the `react` nodes' context.

A `node_event` callback prints progress as nodes start and finish — watch the parallel layer
(`stats`, `charts`, `db_analyst`) light up together.

> **Expect ~45s** with `qwen/qwen3.5-9b`. A reference run produced **7 nodes ok, 0 failed**; its
> output is committed as [`SAMPLE_REPORT.md`](capstone/SAMPLE_REPORT.md) and
> [`sample_dashboard.png`](capstone/sample_dashboard.png) if you want to compare without running.

In [ ]:
question = ("Analyse our sales performance over the period. Which regions and product "
            "categories drive revenue, what is the overall trend, and are there any unusual "
            "spikes or anomalies worth investigating?")

executor = GraphExecutor(graph, provider=provider, native_tools=pool,
                         tool_ctx=ctx, parallelism=3)

t0 = time.time()
def node_event(nid, status, *a):
    print(f"  [node] {nid:12} {status}  (+{time.time()-t0:.0f}s)", flush=True)

result = executor.run(
    {"user_intent": question, "db_path": str(DB), "artifacts_dir": str(ART)},
    node_event=node_event,
)
print("\n" + "=" * 64)
print(result.execution_summary(), f"  ({time.time()-t0:.0f}s)")
for nid, nr in result.nodes.items():
    st = "ok" if nr.ok else ("skip" if nr.skipped else "ERR")
    print(f"  {nid:12} [{st}] {nr.duration_ms} ms")
    if not nr.ok and nr.error:
        print(f"       error: {nr.error}")

---

## 9. Inspect the outputs

Each node's output is on `result.nodes[id].raw_output`. Let's peek at the three LLM nodes — the
SQL findings, what the model *saw* in the dashboard, and the synthesised summary.

In [ ]:
for nid in ("db_analyst", "vision", "synthesis"):
    print(f"\n----- {nid} -----")
    print(str(result.nodes[nid].raw_output)[:800])

Render the dashboard the `vision` node actually looked at:

In [ ]:
from IPython.display import Image, Markdown, display

display(Image(filename=str(ART / "dashboard.png")))

And the final deliverable — `REPORT.md`, assembled by the `report` function node:

In [ ]:
print(result.final.get("report"))          # the report node's status line
display(Markdown((ART / "REPORT.md").read_text()))

When you're done, disconnect from the MCP server (stops the subprocess and the
background session thread):


In [ ]:
session.close()
print("MCP session closed.")


---

## Summary

You built a **seven-node graph** that turns a raw database into a polished insight report —
combining every layer of the framework:

| Piece | Where | Tutorial |
|---|---|---|
| Provider (local, vision forced on) | §1 | [01](01_providers_and_agents.ipynb) |
| Custom tools as `function` nodes | §4 | [02](02_custom_tools.ipynb) |
| DAG + parallel layer, `GraphExecutor` | §7–8 | [03](03_graph_agents.ipynb) |
| `react` node over a live MCP server | §5 | [04](04_mcp_servers.ipynb) |
| **Vision** — a node that reads a chart | §6–7 | this notebook |

### The five integration gotchas (the real lessons)

Anyone wiring these layers together hits these. The capstone code quietly solves all five:

| # | Gotcha | Fix |
|---|---|---|
| 1 | **MCP sessions are event-loop-bound** — the executor's per-node loops make a shared session hang. | Keep the session on one background loop; marshal calls with `run_coroutine_threadsafe`. **Bridge `run()`, not just `call()`.** |
| 2 | **`python_exec` is a throwaway sandbox** — files it writes never reach `cwd`. | Render charts from an in-process **`function` node** to a known `artifacts/` path. |
| 3 | **matplotlib in a worker thread** trips an IPython backend hook. | **Warm up** matplotlib on the main thread; use the OO API (`Figure` + `FigureCanvasAgg`), no global `pyplot`. |
| 4 | **`react` + `finish` → empty `final_text`** — the graph reads the text, not `result.report`. | Tell analysis nodes to **answer in plain text, no `finish`**; cap `max_new_tokens`. |
| 5 | **Vision is off for unknown local models; many images = slow.** | Force `supports_vision=True`; read **one** combined dashboard image. |

### Key ideas

- A graph mixes **deterministic `function` nodes** and **LLM `react` nodes** freely — use code where
  code is right (stats, charts, file I/O) and an agent where judgement is needed (SQL, vision, prose).
- **Parallelism is free**: nodes with no edge between them run concurrently (`parallelism=3` here).
- To give a node **live MCP tools**, drive `GraphExecutor` directly with an explicit `ToolPool` —
  `WorkflowRunner` deliberately can't see them.
- **Vision** is just another tool result: `read_file` an image into a vision-capable provider and the
  model reasons over the pixels.

### What's next

- **[01 — Providers & Agents](01_providers_and_agents.ipynb):** the agent loop these nodes are built on.
- **[03 — Graph Agents](03_graph_agents.ipynb):** the graph/workflow layer, in depth.
- **[04 — MCP Servers](04_mcp_servers.ipynb):** more on connecting and gating external tools.
- **Make it yours:** point `build_dataset` at a real export, swap the MCP server for a Postgres one,
  or add a node — the wiring stays the same.